In [119]:
api_key = 'PKNCYY8KWCWUHRNLPTMM'
secret_key = 'RDH3TDzhdH63rcugihsjl2zcNNs6gwEohCaRy51n'

https://github.com/alpacahq/alpaca-py/blob/master/docs/market_data.rst \
Create instance for clients, pass requests to client\
\
Trading or Broker Clients exist. Requires Broker/Trading API keys\
\
Market Data API allows access to data
- Historical Data Clients, alpaca.data.historical -> Stock, Options, Crypto
    - No API Key required for Crypto
    - Passing requests to clients -> alpaca.data/trading.requests
      - Response params (data):
        - Symbol -> [strs]
        - timeframe -> frequency -> alpaca.datetime.timeframe.TimeFrame(amount=, unit=TimeFrameUnit.minute/hour.second)
        - start
        - end (optional = now default)
        - limit = int
    - Response to df -> .df

        

In [120]:
#Trading or broker client
from alpaca.trading.client import TradingClient

#Stocks, options or crypto
from alpaca.data.historical.stock import StockHistoricalDataClient


from alpaca.trading.stream import TradingStream
from alpaca.data.live.stock import StockDataStream

In [121]:
from alpaca.data.requests import (
    CorporateActionsRequest,
    StockBarsRequest,
    StockQuotesRequest,
    StockTradesRequest,
)
from alpaca.trading.requests import (
    ClosePositionRequest,
    GetAssetsRequest,
    GetOrdersRequest,
    LimitOrderRequest,
    MarketOrderRequest,
    StopLimitOrderRequest,
    StopLossRequest,
    StopOrderRequest,
    TakeProfitRequest,
    TrailingStopOrderRequest,
)
from alpaca.trading.enums import (
    AssetExchange,
    AssetStatus,
    OrderClass,
    OrderSide,
    OrderType,
    QueryOrderStatus,
    TimeInForce,
)

In [122]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit

In [123]:
tc = TradingClient(api_key,secret_key)
#tc.get_account()
sdc = StockHistoricalDataClient(api_key, secret_key)

In [125]:
now = datetime.now(ZoneInfo("America/New_York"))

req = StockBarsRequest(
    symbol_or_symbols = [symbol],
    timeframe=TimeFrame(amount = 15, unit = TimeFrameUnit.Minute), # specify timeframe
    start = now - timedelta(days = 5),                             # specify start datetime, default=the beginning of the current day.
    #end_date=None,                                                # specify end datetime, default=now
    limit = 2,                                                    # specify limit
)
sdc.get_stock_bars(req).df

open   high     low  close    volume  \
symbol timestamp                                                          
TQQQ   2025-01-02 12:15:00+00:00  82.00  82.11  81.920  82.01  102382.0   
       2025-01-02 12:30:00+00:00  82.04  82.05  81.601  81.66  133828.0   

                                  trade_count       vwap  
symbol timestamp                                          
TQQQ   2025-01-02 12:15:00+00:00       1059.0  82.010638  
       2025-01-02 12:30:00+00:00       1154.0  81.898016

In [126]:
# get historical trades by symbol
req = StockTradesRequest(
    symbol_or_symbols = [symbol],
    start = now - timedelta(days = 5),                          # specify start datetime, default=the beginning of the current day.
    # end=None,                                             # specify end datetime, default=now
    limit = 2,                                                # specify limit
)
sdc.get_stock_trades(req).df

exchange  price  size    id  \
symbol timestamp                                                      
TQQQ   2025-01-02 12:11:00.948935+00:00        Z  81.95   2.0   157   
       2025-01-02 12:11:00.951689+00:00        K  81.95   3.0  4444   

                                           conditions tape  
symbol timestamp                                            
TQQQ   2025-01-02 12:11:00.948935+00:00  [@, F, T, I]    C  
       2025-01-02 12:11:00.951689+00:00  [@, F, T, I]    C

In [104]:
# get historical quotes by symbol
req = StockQuotesRequest(
    symbol_or_symbols = [symbol],
    start = now - timedelta(days = 5),                      # specify start datetime, default=the beginning of the current day.
    # end=None,                                             # specify end datetime, default=now
    limit = 2,                                              # specify limit
)
sdc.get_stock_quotes(req).df

bid_price  bid_size bid_exchange  \
symbol timestamp                                                            
TQQQ   2025-01-02 09:00:00.001667+00:00      80.84       9.0            Q   
       2025-01-02 09:00:00.001844+00:00      80.84       9.0            Q   

                                         ask_price  ask_size ask_exchange  \
symbol timestamp                                                            
TQQQ   2025-01-02 09:00:00.001667+00:00       0.00       0.0                
       2025-01-02 09:00:00.001844+00:00      80.88       9.0            Q   

                                        conditions tape  
symbol timestamp                                         
TQQQ   2025-01-02 09:00:00.001667+00:00        [Y]    C  
       2025-01-02 09:00:00.001844+00:00        [R]    C

In [127]:
# get latest quotes by symbol
req = StockQuotesRequest(
    symbol_or_symbols = [symbol],
)
res = sdc.get_stock_latest_quote(req)
res

{'TQQQ': {   'ask_exchange': 'V',
     'ask_price': 86.0,
     'ask_size': 17.0,
     'bid_exchange': 'V',
     'bid_price': 85.03,
     'bid_size': 17.0,
     'conditions': ['R'],
     'symbol': 'TQQQ',
     'tape': 'C',
     'timestamp': datetime.datetime(2025, 1, 6, 21, 59, 55, 8559, tzinfo=TzInfo(UTC))}}

In [128]:
import nest_asyncio
nest_asyncio.apply()

In [139]:
stock_data_stream_client = StockDataStream(api_key, secret_key)

async def stock_data_stream_handler(data):
    print(data)

symbols = ['TSLA']

stock_data_stream_client.subscribe_quotes(stock_data_stream_handler, *symbols)
stock_data_stream_client.subscribe_trades(stock_data_stream_handler, *symbols)

stock_data_stream_client.run()

symbol='TSLA' timestamp=datetime.datetime(2025, 1, 7, 14, 44, 10, 581359, tzinfo=datetime.timezone.utc) bid_price=410.0 bid_size=3.0 bid_exchange='V' ask_price=413.01 ask_size=2.0 ask_exchange='V' conditions=['R'] tape='C'
symbol='TSLA' timestamp=datetime.datetime(2025, 1, 7, 14, 44, 10, 581516, tzinfo=datetime.timezone.utc) bid_price=410.0 bid_size=3.0 bid_exchange='V' ask_price=416.3 ask_size=1.0 ask_exchange='V' conditions=['R'] tape='C'
symbol='TSLA' timestamp=datetime.datetime(2025, 1, 7, 14, 44, 10, 583202, tzinfo=datetime.timezone.utc) bid_price=412.72 bid_size=1.0 bid_exchange='V' ask_price=416.3 ask_size=1.0 ask_exchange='V' conditions=['R'] tape='C'
symbol='TSLA' timestamp=datetime.datetime(2025, 1, 7, 14, 44, 10, 584489, tzinfo=datetime.timezone.utc) bid_price=410.0 bid_size=3.0 bid_exchange='V' ask_price=416.3 ask_size=1.0 ask_exchange='V' conditions=['R'] tape='C'
symbol='TSLA' timestamp=datetime.datetime(2025, 1, 7, 14, 44, 11, 146208, tzinfo=datetime.timezone.utc) bid_pr

TimeoutError: 

In [138]:
from alpaca.data.historical.news import NewsClient
from alpaca.data.requests import NewsRequest
from datetime import datetime

# no keys required for news data
client = NewsClient(api_key, secret_key)

request_params = NewsRequest(
                        symbols="TSLA",
                        start=datetime.strptime("2024-12-31", '%Y-%m-%d')
                        )

news = client.get_news(request_params)

# convert to dataframe
news.df

,headline,source,url,summary,created_at,updated_at,symbols,author,content,images
id,,,,,,,,,,
42845120,Top 10 Trending Stocks On WallStreetBets As Of...,benzinga,https://www.benzinga.com/trading-ideas/25/01/4...,,2025-01-07 14:17:35+00:00,2025-01-07 14:17:35+00:00,"[AAPL, AMD, CVNA, DIS, FUBO, META, MU, NVDA, T...",Benzinga Newsdesk,,[]
42843580,"Acelyrin, Tesla And Other Big Stocks Moving Lo...",benzinga,https://www.benzinga.com/trading-ideas/movers/...,,2025-01-07 13:42:11+00:00,2025-01-07 13:42:12+00:00,"[COGT, NPCE, SLRN, THRD, TKNO, TSLA, YMAB, ZJK]",Avi Kapoor,,"[{'size': 'NewsImageSize.LARGE', 'url': 'https..."
42842267,Trump's SEC Pick Likely To 'Bring Down The Str...,benzinga,https://www.benzinga.com/25/01/42842267/trumps...,"Atkins, a former SEC commissioner, is known fo...",2025-01-07 13:10:41+00:00,2025-01-07 13:10:42+00:00,"[COIN, QQQ, SPY, TSLA]",Pooja Rajkumari,,"[{'size': 'NewsImageSize.LARGE', 'url': 'https..."
42840097,NHTSA Opens Probe Into 2.6 Million Tesla Vehic...,benzinga,https://www.benzinga.com/25/01/42840097/nhtsa-...,The NHTSA on Tuesday opened an investigation i...,2025-01-07 12:00:02+00:00,2025-01-07 12:00:02+00:00,[TSLA],Anan Ashraf,,"[{'size': 'NewsImageSize.LARGE', 'url': 'https..."
42839105,Tesla Reminisces Delivering First Model 3 From...,benzinga,https://www.benzinga.com/25/01/42839105/tesla-...,EV giant Tesla on Monday reminisced about deli...,2025-01-07 11:11:10+00:00,2025-01-07 11:11:10+00:00,[TSLA],Anan Ashraf,,"[{'size': 'NewsImageSize.LARGE', 'url': 'https..."
...,...,...,...,...,...,...,...,...,...,...
42737037,Tesla Shanghai Megafactory Starts Trial Produc...,benzinga,https://www.benzinga.com/news/global/24/12/427...,Tesla said that its megapack factory in Shangh...,2024-12-31 08:55:39+00:00,2024-12-31 08:55:39+00:00,[TSLA],Anan Ashraf,,"[{'size': 'NewsImageSize.LARGE', 'url': 'https..."
42736503,"Dow Tumbles Over 400 Points As Tesla, Meta Dec...",benzinga,https://www.benzinga.com/24/12/42736503/dow-tu...,,2024-12-31 07:25:41+00:00,2024-12-31 07:25:41+00:00,"[META, TSLA]",Avi Kapoor,,"[{'size': 'NewsImageSize.LARGE', 'url': 'https..."
42735573,"Palantir, Salesforce, And Snowflake Made The C...",benzinga,https://www.benzinga.com/24/12/42735573/two-ma...,Wedbush analyst Dan Ives revealed his top 10 t...,2024-12-31 02:43:45+00:00,2024-12-31 02:43:45+00:00,"[AAPL, AMZN, CRM, GOOG, GOOGL, MDB, META, MSFT...",Ananya Gairola,,"[{'size': 'NewsImageSize.LARGE', 'url': 'https..."
